# Chapter 2 Lab — Data Acquisition and Preprocessing

Covers all four acquisition paths (OCR, ASR, scraping, already-digital text) on small,
self-contained inputs, then builds a TF-IDF + PCA pipeline. No live network calls and no
third-party copyrighted data are used, so the notebook is fully reproducible offline.

## 1. OCR — image → text

We synthesize a text image ourselves with PIL, so there is no scanned copyrighted document
involved. Requires `pytesseract` + the Tesseract binary installed separately (not bundled here).

In [ ]:
from PIL import Image, ImageDraw, ImageFont

img = Image.new("RGB", (400, 60), color="white")
draw = ImageDraw.Draw(img)
draw.text((10, 15), "Modern NLP Systems", fill="black")
img.save("synthetic_ocr_sample.png")
img

In [ ]:
try:
    import pytesseract
    text = pytesseract.image_to_string(Image.open("synthetic_ocr_sample.png"))
    print(repr(text))
except Exception as e:
    print("Tesseract binary not installed in this environment — see README for setup.", e)

## 2. ASR — speech → text

We synthesize a short public-domain phrase with a local text-to-speech engine (no copyrighted
recording bundled), then transcribe it back.

In [ ]:
try:
    import pyttsx3
    engine = pyttsx3.init()
    engine.save_to_file("Practical natural language processing with Python.", "synthetic_speech.wav")
    engine.runAndWait()
    print("synthetic_speech.wav written")
except Exception as e:
    print("TTS engine unavailable in this environment — see README for setup.", e)

## 3. Scraping — a local, static HTML fixture (not a live site)

Scraping a live website inside a shipped notebook breaks reproducibility (layout changes, rate
limits, ToS). We scrape a tiny local HTML fixture instead — the technique transfers directly to
a real site's HTML, saved responsibly via an API or with permission.

In [ ]:
from bs4 import BeautifulSoup

fixture_html = """
<html><body>
  <h1>Sample Page</h1>
  <p class="content">This is a locally hosted fixture used only to demonstrate scraping.</p>
</body></html>
"""
soup = BeautifulSoup(fixture_html, "html.parser")
print(soup.h1.text)
print(soup.find("p", class_="content").text)

## 4. TF-IDF + PCA on a small public sample

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

headlines = [
    "Local team wins championship in dramatic final",
    "New research links exercise to better sleep",
    "City council approves new park downtown",
    "Championship parade draws record crowd",
    "Study finds link between diet and heart health",
]
X = TfidfVectorizer().fit_transform(headlines).toarray()
coords = PCA(n_components=2).fit_transform(X)

plt.scatter(coords[:, 0], coords[:, 1])
for i, h in enumerate(headlines):
    plt.annotate(h[:20] + "...", (coords[i, 0], coords[i, 1]), fontsize=8)
plt.title("TF-IDF headlines projected to 2D with PCA")
plt.show()

## Exercise

Replace `headlines` with ten sentences from your own domain of interest. Do the PCA clusters
still separate topics sensibly? What does that tell you about TF-IDF's reliance on shared
vocabulary rather than meaning?